In [1]:
!pip install boto3 psycopg2-binary pgvector sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 3.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 5.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 6.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 6.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.0/146.0 MB 6.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 4.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 6.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.6 MB/s et

In [2]:
import boto3

# Connect to MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password123"
)

# Create a sample document
sample_text = """
JPMorgan Chase is a leading global financial services firm.
The company provides investment banking, financial services for consumers,
small business and commercial banking, financial transaction processing,
asset management and private banking services.
"""

# Upload to MinIO
s3.put_object(
    Bucket="raw-data",
    Key="documents/jpmorgan.txt",
    Body=sample_text.encode("utf-8")
)

print("File uploaded successfully!")

File uploaded successfully!


In [3]:
from sentence_transformers import SentenceTransformer

# Read file back from MinIO
response = s3.get_object(Bucket="raw-data", Key="documents/jpmorgan.txt")
text = response["Body"].read().decode("utf-8")
print("Retrieved text:", text)

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Chunk the text into sentences
chunks = [s.strip() for s in text.split(".") if s.strip()]
print("\nChunks:", chunks)

# Create embeddings
embeddings = model.encode(chunks)
print("\nEmbedding shape:", embeddings.shape)
print("First embedding (first 5 values):", embeddings[0][:5])

Retrieved text: 
JPMorgan Chase is a leading global financial services firm.
The company provides investment banking, financial services for consumers,
small business and commercial banking, financial transaction processing,
asset management and private banking services.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Chunks: ['JPMorgan Chase is a leading global financial services firm', 'The company provides investment banking, financial services for consumers,\nsmall business and commercial banking, financial transaction processing,\nasset management and private banking services']

Embedding shape: (2, 384)
First embedding (first 5 values): [-0.00292211 -0.01355885 -0.03309703  0.03606996 -0.00911433]


In [4]:
#Now let's store these embeddings in pgvector. Paste this in the next cell:

In [5]:
import psycopg2
from pgvector.psycopg2 import register_vector

# Connect to pgvector
conn = psycopg2.connect(
    host="pgvector",
    port=5432,
    database="vectordb",
    user="admin",
    password="password123"
)
cursor = conn.cursor()

# Enable pgvector extension
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)

# Create table to store embeddings
cursor.execute("""
    CREATE TABLE IF NOT EXISTS documents (
        id SERIAL PRIMARY KEY,
        content TEXT,
        embedding vector(384)
    );
""")
conn.commit()

# Insert chunks and their embeddings
for chunk, embedding in zip(chunks, embeddings):
    cursor.execute(
        "INSERT INTO documents (content, embedding) VALUES (%s, %s)",
        (chunk, embedding.tolist())
    )

conn.commit()
print("Embeddings stored successfully!")

Embeddings stored successfully!


In [6]:
#Now let's query the vector database to find similar documents. Paste this in the next cell:

In [7]:
# Search for similar documents
query = "What services does JPMorgan provide?"

# Convert query to embedding
query_embedding = model.encode([query])[0]

# Search pgvector for most similar chunks
cursor.execute("""
    SELECT content, 1 - (embedding <=> %s::vector) AS similarity
    FROM documents
    ORDER BY similarity DESC
    LIMIT 2;
""", (query_embedding.tolist(),))

results = cursor.fetchall()
print("Query:", query)
print("\nTop matching chunks:")
for i, (content, similarity) in enumerate(results):
    print(f"\n{i+1}. Similarity: {similarity:.4f}")
    print(f"   Content: {content}")

Query: What services does JPMorgan provide?

Top matching chunks:

1. Similarity: 0.7425
   Content: JPMorgan Chase is a leading global financial services firm

2. Similarity: 0.5658
   Content: The company provides investment banking, financial services for consumers,
small business and commercial banking, financial transaction processing,
asset management and private banking services
